##**Load Data**

In [2]:
import pandas as pd

In [4]:
reviews_df =  pd.read_csv('classified_reviews.csv')

In [3]:
clustered_products = pd.read_csv('clustered_products.csv')


## **Merge the clustered product data with the reviews**


In [7]:
# 1. Merge the clustered product data with the reviews
merged_data = reviews_df.merge(clustered_products[['name', 'cluster_labels', 'cluster_name']],
                              on='name', how='left')

In [8]:
# Check if any reviews didn't get matched to clusters
unmatched = merged_data[merged_data['cluster_labels'].isna()]
print(f"Number of unmatched reviews: {len(unmatched)}")

# Save the merged data for Task 3
merged_data.to_csv('reviews_with_clusters.csv', index=False)

Number of unmatched reviews: 0


In [10]:
cluster_names= {'Cluster 0': 'Batteries',
'Cluster 1': 'Ebook readers',
'Cluster 2': 'Electronics',
'Cluster 3': 'Accessories',
'Cluster 4': 'Home & Office' }

##  **For each cluster, identify top and worst products based on ratings**

In [17]:
def analyze_cluster(cluster_id, cluster_name, data):
    # Filter data for this cluster
    cluster_data = data[data['cluster_labels'] == cluster_id]

    # Print debug info
    print(f"Analyzing cluster {cluster_id} ({cluster_name})")
    print(f"Found {len(cluster_data)} items in this cluster")


    # Calculate average ratings per product
    product_ratings = cluster_data.groupby('name').agg(
        avg_rating=('reviews.rating','mean'),
        count=('reviews.rating','count')
    ).reset_index()

    # Only consider products with sufficient reviews (e.g., at least 5)
    product_ratings = product_ratings[product_ratings['count'] >= 5]
    print(f"Found {len(product_ratings)} products with sufficient reviews")

    # Get top 3 products
    top_products = product_ratings.sort_values('avg_rating', ascending=False).head(3)

    # Get worst product
    worst_product = product_ratings.sort_values('avg_rating', ascending=True).head(1)

    return {
        'cluster_id': cluster_id,
        'cluster_name': cluster_name,
        'top_products': top_products,
        'worst_product': worst_product
    }

In [20]:
# Analyze each cluster properly matching the integer IDs
cluster_analyses = []
for cluster_id in merged_data['cluster_labels'].unique():
    cluster_name = cluster_names.get(cluster_id, f"Cluster {cluster_id}")
    analysis = analyze_cluster(cluster_id, cluster_name, merged_data)
    cluster_analyses.append(analysis)

Analyzing cluster 2 (Cluster 2)
Found 6737 items in this cluster
Found 19 products with sufficient reviews
Analyzing cluster 3 (Cluster 3)
Found 312 items in this cluster
Found 12 products with sufficient reviews
Analyzing cluster 4 (Cluster 4)
Found 457 items in this cluster
Found 2 products with sufficient reviews
Analyzing cluster 1 (Cluster 1)
Found 14661 items in this cluster
Found 27 products with sufficient reviews
Analyzing cluster 0 (Cluster 0)
Found 12071 items in this cluster
Found 2 products with sufficient reviews


##  **Extract key insights for each product**



In [29]:
def extract_product_insights(product_name, data, sentiment='Positive'):

    sentiment = sentiment.capitalize()

    # Check if product exists
    if product_name not in data['name'].values:
        return []

    # Filter reviews for this product
    product_reviews = data[data['name'] == product_name]

    if sentiment not in ['Positive', 'Negative', 'Neutral']:
        return []

    sentiment_reviews = product_reviews[product_reviews['sentiment'] == sentiment]

    # Check if have reviews with the requested sentiment
    if len(sentiment_reviews) == 0:
        return []

    # Extract the review texts
    review_texts = sentiment_reviews['reviews.text'].dropna().tolist()
    review_texts = [r.strip() for r in review_texts if isinstance(r, str) and r.strip()]

    return review_texts


## **Loading pre-trained model for text summarization**

In [ ]:
from transformers import pipeline

# Initialize summarizer
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

## **Generate article content**

In [30]:
def generate_article(cluster_analysis, merged_data):
   # Get the cluster information
    cluster_id = cluster_analysis['cluster_id']
    cluster_mapping = {'Cluster 0': 'Batteries', 'Cluster 1': 'Ebook readers', 'Cluster 2': 'Electronics',
                       'Cluster 3': 'Accessories', 'Cluster 4': 'Home & Office'}

    cluster_name = cluster_mapping.get(f'Cluster {cluster_id}', f'Category {cluster_id}')

    top_products = cluster_analysis['top_products']['name'].tolist()
    worst_product = cluster_analysis['worst_product']['name'].iloc[0] if not cluster_analysis['worst_product'].empty else None

    article = f"# Product Recommendations for {cluster_name}\n\n"

    # Top products section
    article += "## Top 3 Recommended Products\n\n"

    for product in top_products:
        # Get positive reviews for this product
        positive_reviews = extract_product_insights(product, merged_data, 'Positive')

        # Get negative reviews (complaints)
        negative_reviews = extract_product_insights(product, merged_data, 'Negative')

        # Better filtering for meaningful reviews
        positive_reviews = [review for review in positive_reviews if isinstance(review, str) and
                           review.lower() != 'none' and len(review.split()) > 5]
        negative_reviews = [review for review in negative_reviews if isinstance(review, str) and
                           review.lower() != 'none' and len(review.split()) > 5]

        # Check if we have enough reviews for summarization
        if positive_reviews:
            positive_text = " ".join(positive_reviews[:10])  # Use first 10 positive reviews

            # Only use summarizer if we have substantial text
            if len(positive_text.split()) > 30:
                try:
                    pos_summary = summarizer(positive_text, max_length=100, min_length=30, do_sample=False)[0]['summary_text']
                except Exception as e:
                    print(f"Error summarizing positive reviews for {product}: {e}")
                    pos_summary = positive_reviews[0] if positive_reviews else "No detailed positive feedback available."
            else:
                # If not enough text for summarization, use the most helpful review
                pos_summary = positive_reviews[0] if positive_reviews else "No detailed positive feedback available."
        else:
            # Check if there are any positive ratings without text
            product_ratings = merged_data[(merged_data['name'] == product) &
                                         (merged_data['reviews.rating'] >= 4)]
            if len(product_ratings) > 0:
                pos_summary = f"Customers rate this product highly with an average of {product_ratings['reviews.rating'].mean():.1f}/5 stars based on {len(product_ratings)} ratings."
            else:
                pos_summary = "Limited positive feedback available for this product."

        # Similar approach for negative reviews
        if negative_reviews:
            negative_text = " ".join(negative_reviews[:10])

            if len(negative_text.split()) > 30:
                try:
                    neg_summary = summarizer(negative_text, max_length=100, min_length=30, do_sample=False)[0]['summary_text']
                except Exception as e:
                    print(f"Error summarizing negative reviews for {product}: {e}")
                    neg_summary = negative_reviews[0] if negative_reviews else "No detailed complaints found."
            else:
                neg_summary = negative_reviews[0] if negative_reviews else "No detailed complaints found."
        else:
            # Check if there are any negative ratings without text
            product_ratings = merged_data[(merged_data['name'] == product) &
                                         (merged_data['reviews.rating'] <= 2)]
            if len(product_ratings) > 0:
                neg_summary = f"Some customers gave lower ratings (average {product_ratings['reviews.rating'].mean():.1f}/5) but didn't provide detailed feedback."
            else:
                neg_summary = "Few or no complaints found for this product."

        # Add to article
        article += f"### {product}\n"
        article += f"**What customers love:** {pos_summary}\n\n"
        article += f"**Common complaints:** {neg_summary}\n\n"

    # [rest of the function remains the same]
        # Compare the products
    article += "## Key Differences Between Top Products\n\n"
    # This would ideally be generated from a more sophisticated analysis
    # For now, we'll use a simple placeholder
    article += "Based on customer reviews, these products differ in terms of durability, ease of use, and value for money.\n\n"

    # Worst product section
    if worst_product:
        article += "## Product to Avoid\n\n"

        # Get negative reviews for worst product
        negative_reviews = extract_product_insights(worst_product, merged_data, 'Negative')
        negative_text = " ".join(negative_reviews[:10])  # Use first 10 negative reviews

        if negative_text:
            avoid_summary = summarizer(negative_text, max_length=120, min_length=40, do_sample=False)[0]['summary_text']
        else:
            avoid_summary = "Limited data available, but customer ratings suggest this product underperforms compared to alternatives."

        article += f"### {worst_product}\n"
        article += f"**Why you should avoid it:** {avoid_summary}\n"

    return article

# Generate articles for all clusters
for analysis in cluster_analyses:
    article = generate_article(analysis, merged_data)

    # Save to file
    with open(f"article_cluster_{analysis['cluster_id']}.md", "w") as f:
        f.write(article)

    print(f"Generated article for {analysis['cluster_name']}")

Device set to use cpu


Generated article for Cluster 2
Generated article for Cluster 3
Generated article for Cluster 4


Your max_length is set to 100, but your input_length is only 67. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)


Generated article for Cluster 1
Generated article for Cluster 0
